## 04. scratchpad

### 1. Data Upload

In [1]:
import pandas as pd
import os
from collections import defaultdict, Counter
import re
import time
from tqdm.notebook import tqdm
import copy

In [2]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [3]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [4]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [5]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [6]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [7]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [8]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
1195,On the avenue,¶ Na alei
349,- I knew it.,Maréchal nie mógł się na to zdobyć.
11,"Yoko, a famous singer, had married Koichiro an...",Słynna śpiewaczka Yôko poślubiła Kôichirô i po...
484,I see you had a little publicity...,"Widze, že masz juž reklame."
619,I've been told the whole thing is his idea to ...,"Slyszalem, že wymyšlil szwindel, by sie wzboga..."
142,"My, you ought to see how little Emily's grown.","Ojej, powinieneś zobaczyć jak ​​mała Emily uro..."
88,since you have at last renounced the errors of...,"Nie spotka cię ekskomunika, albowiem przyznała..."
379,- I don't know... Maybe.,- Nie wiem... može.
578,If there's money in each one...,Ješli w každym sa pieniadze?
971,"Hamburg, GroBe Bleichen 1 5.","Hamburg, Große Bleichen 15."


In [9]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
243257,No.,Niestety.
20712,I'll bet he stole those boots!,"Założę się, że ukradł te buty!"
342049,THERE ARE TAX DEDUCTIONS. THERE HAD NEVER BEEN...,Coś nie z tej ziemi.
278998,I'm freezing.,Zimno mi.
196034,"And whatever they've done to him, no matter wh...","I cokolwiek z nim zrobią, nie ważnie gdzie ter..."


In [10]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [11]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [12]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [13]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [14]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [15]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [16]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [17]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(5)

,eng_text,pol_text
32447,Is anyone there?,Czy jest ktoś?
265856,Uh-uh. I should've told you first that I love ya.,"Od razu powinienem ci powiedzieć, że kocham cię."
215180,"No, I don't go no place much.","Nie, nie chodze ma miejsca duzo."
432755,Don't kid me.,Nie oszukuj mnie.
378859,I'm Professor Siletsky.,Jestem profesor Silecki.


#### 2.5 Broken Dialog like sentences [somehwere has -]

# Do naprawy (lepszy re przed tokenziacja)

In [18]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [19]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
5796,I hear in my head the judge's voice as he pron...,"Słyszę w mojej głowie wyrok sądu, który stwier..."
195540,"Hey, Rock! - So long, Rocky!","Na razie, Rocky!"
285559,"""Or What the Kitchen Maid Saw Through the Keyh...","""Albo to, co podejrzała kucharka."" Koniec cytatu."
288557,Mademoiselle will have lunch with me.,Zechce pani zjeść obiad razem ze mną? - Nie mo...
540943,About what?,Myślę o... - Powiedz!


In [20]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [21]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [22]:
df_5.sample(5)

,eng_text,pol_text
301312,No!,Nie!
179544,"Fifty dollars, a hundred dollars. A thousand d...","50 dolarów, 100, 1000 góra."
291552,"Here we are, good strong tea.",O mamy herbatkę.
59836,Get me some too.,Dla mnie też.
474245,She doesn't know anybody here.,Nikogo tu nie zna.


#### 2.6 Different Last Case

In [23]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [24]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [25]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [26]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [27]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [28]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(5)

,eng_text,pol_text,len_ratio
133186,"But you see, I've never had the advantages tha...","Ale widzisz, nigdy nie miałem tego co inni mal...",1.25
102630,"But, madame, the bank would not be interested ...","Ale, proszę pani, bank nie jest zainteresowany...",1.40
75593,Did he tell you where he was going?,Powiedział dokąd się wybiera?,2.00
448097,"She wanted to break the rules, like cigarettes...","Łamała zasady, zaczęła palić, używać perfum, s...",1.40
469816,All I ask is for you to let me love you.,"Jedyne o co cię proszę, to abyś pozwolił mi ci...",1.00


In [29]:
df_7.sample(5)

,eng_text,pol_text,len_ratio
328287,What house?,Gdzie?,2.000000
449604,His car's out there in the rain.,Jego wóz tam stoi.,1.750000
18686,Christian Thompson.,Christian Thompson.,1.000000
20339,"For three days and nights, I've had no other t...","Trzy dni i noce myślałem tylko o tym, jak cię ...",1.272727
29424,or a traitor.,Lub zdrajca.,1.500000


#### 2.8 Sentences starting with * fix

In [30]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [31]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [32]:
df_8.sample(5)

,eng_text,pol_text,len_ratio
284414,"All right, back it up!",Cofnijcie się!,2.500000
110487,"You may be right, Calamity.","Może masz rację, Calamity.",1.250000
65780,"Sir, you will address Her Highness by her corr...",Proszę zwracać się do Jej Wysokości po właściw...,1.111111
103889,I want to confess!,Chcę się przyznać!,1.333333
416846,Nobody knows me around here.,Nikt mnie tu nie zna.,1.000000


#### 2.10  " ` " ---> " ' " & del " ~ "

In [33]:
wanted_case = '`'
mask = df_8["eng_text"].apply(lambda x: wanted_case in x)
df_8[mask]

,eng_text,pol_text,len_ratio
214145,"Revolution had broken out, her diplomats sued ...","Rewolucja na tyłach frontu, dąży do zawarcia p...",1.277778
214152,We`ll examine it.,Zbadajmy go.,1.500000
214157,What do you think you`re doing?,Wstawaj! Co ty robisz?,1.500000
214161,Where`s your hand grenade?,Gdzie masz swój granat?,1.000000
214163,Let `em have it!,Niech je poczują!,1.333333
...,...,...,...
214943,We don`t want to hate one another.,Bez nienawiści ani pogardy.,1.750000
214944,There`s room for everyone.,Każdy ma swe miejsce.,1.000000
214947,"Greed has poisoned men`s souls, has barricaded...","Chciwość zatruła nasze dusze, wzniosła mury ni...",1.333333
214963,You don`t hate.,Przestańcie nienawidzieć.,1.500000


In [34]:
df_8["eng_text"] = df_8["eng_text"].str.replace("`", "'", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("`", "'", regex=False)

In [35]:
df_8["eng_text"] = df_8["eng_text"].str.replace("~", "", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("~", "", regex=False)

#### 2.11 Del rows with specific case

In [36]:
def del_row_with_char(df, eng_col, pol_col, chars):
    df_proc = df.copy()
    for char in chars:
        mask1 = df_proc[eng_col].apply(lambda x: char in x)
        mask2 = df_proc[pol_col].apply(lambda x: char in x)
        df_proc = df_proc[~(mask1 | mask2)].reset_index(drop=True)
    return df_proc
    

chars = "=+*@#;|_}{"
df_9 = del_row_with_char(df_8, "eng_text", "pol_text", chars)

#### 2.12 Del rows with dialog -

In [37]:
mask1 = df_9["eng_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
mask2 = df_9["pol_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
df_data = df_9[~(mask1 | mask2)].reset_index(drop=True)

#### 3. Data Tokenization

In [38]:
uniq_cases = set("".join(df_data["eng_text"].astype(str)))
tab1 = sorted(list(uniq_cases), reverse=True)
print(tab1)

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [39]:
uniq_cases = set("".join(df_data["pol_text"].astype(str)))
tab2 = sorted(list(uniq_cases), reverse=True)
print(tab2)

['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [40]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [41]:
def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [42]:
df_data = df_data.drop(['len_ratio'], axis=1)

#### 3.1 Splitting and trimming on quantile

In [43]:
df_data['eng_split'] = df_data['eng_text'].apply(tokenize_eng)
df_data['pol_split'] = df_data['pol_text'].apply(tokenize_pol)
df_data.sample(10)

,eng_text,pol_text,eng_split,pol_split
428269,I don't know the man.,Nie znam go.,"[i, don, 't, know, the, man, .]","[nie, znam, go, .]"
351732,Nobody was proud of me.,Nikt nie był ze mnie dumny.,"[nobody, was, proud, of, me, .]","[nikt, nie, był, ze, mnie, dumny, .]"
38069,Yes. My mother was Egyptian.,Moja mama była Egipcjanką.,"[yes, ., my, mother, was, egyptian, .]","[moja, mama, była, egipcjanką, .]"
278164,I'll drop by in the morning.,"Spadam już, bo się rozwidnia.","[i, 'll, drop, by, in, the, morning, .]","[spadam, już, ,, bo, się, rozwidnia, .]"
359066,Come on!,Chodźcie!,"[come, on, !]","[chodźcie, !]"
11235,"Well, get a load of this, big boy.","Dobra, posłuchaj mnie, grubasie.","[well, ,, get, a, load, of, this, ,, big, boy, .]","[dobra, ,, posłuchaj, mnie, ,, grubasie, .]"
15414,A dagger.,Sztylet.,"[a, dagger, .]","[sztylet, .]"
194953,Sure.,Pewnie.,"[sure, .]","[pewnie, .]"
237836,"Good morning, Miss Bragg.","Dzień dobry, panno Bragg.","[good, morning, ,, miss, bragg, .]","[dzień, dobry, ,, panno, bragg, .]"
348275,You're with me.,Jesteś ze mną.,"[you, 're, with, me, .]","[jesteś, ze, mną, .]"


# TODO - zamien lt na It

In [44]:
df_data[df_data['eng_split'].apply(lambda x: 'lt' in x)]

,eng_text,pol_text,eng_split,pol_split
9781,lt was the only thing they had.,"To było jedyne, jakie mieli wolne.","[lt, was, the, only, thing, they, had, .]","[to, było, jedyne, ,, jakie, mieli, wolne, .]"
9811,lt was your fault you put the thing under the ...,"To twoja wina, bo położyłeś go pod łóżko.","[lt, was, your, fault, you, put, the, thing, u...","[to, twoja, wina, ,, bo, położyłeś, go, pod, ł..."
10953,Beat it. lt's a bull!,"Uciekajmy, to gliniarz!","[beat, it, ., lt, 's, a, bull, !]","[uciekajmy, ,, to, gliniarz, !]"
10957,lt went in your pocket.,Wpadł ci do kieszeni.,"[lt, went, in, your, pocket, .]","[wpadł, ci, do, kieszeni, .]"
10981,lt caught in your pocket.,On utknął w kieszeni...,"[lt, caught, in, your, pocket, .]","[on, utknął, w, kieszeni, ., ., .]"
...,...,...,...,...
412362,Okay. lt's your f uneral.,Dobrze. To twój cyrk.,"[okay, ., lt, 's, your, f, uneral, .]","[dobrze, ., to, twój, cyrk, .]"
412366,lt's important.,To ważne.,"[lt, 's, important, .]","[to, ważne, .]"
412439,"Maybe, maybe not. lt's a pretty story.","Może tak, może nie. Niezła historia.","[maybe, ,, maybe, not, ., lt, 's, a, pretty, s...","[może, tak, ,, może, nie, ., niezła, historia, .]"
412454,lt's unlocked.,Jest otwarta.,"[lt, 's, unlocked, .]","[jest, otwarta, .]"


In [45]:
thres1 = df_data['eng_split'].str.len().quantile(0.99)
mask1 = df_data['eng_split'].str.len() > thres1

thres2 = df_data['pol_split'].str.len().quantile(0.99)
mask2 = df_data['pol_split'].str.len() > thres2
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())

df_data = df_data[~(mask1 | mask2)].reset_index(drop=True)

4113 3960 5510


#### 3.2 BPE Init. Uniq_freq & Rozklad chars

In [46]:
uniq_freq_eng = Counter(df_data['eng_split'].explode())
uniq_freq_pol = Counter(df_data['pol_split'].explode())

In [47]:
len(uniq_freq_eng), len(uniq_freq_pol)

(40087, 113603)

In [48]:
char_factor = lambda word: [x for x in word] + ['_']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

40087 [[['.', '_'], 435802], [[',', '_'], 177291], [['y', 'o', 'u', '_'], 134548], [['i', '_'], 122767], [['t', 'h', 'e', '_'], 89859]] 

113603 [[['.', '_'], 413780], [[',', '_'], 183948], [['?', '_'], 83444], [['n', 'i', 'e', '_'], 82694], [['t', 'o', '_'], 53190]]


#### 3.3 BPE Tokenizer

In [72]:
class BPETokenizer_5():
    def __init__(self, vocab_chrs, vocab_pairs, vocab_size, is_tgt):
        self.vocab_chrs = vocab_chrs
        self.vocab_pairs = vocab_pairs
        self.vocab_size = vocab_size
        self.pair_counts = self.pair_counts_init(vocab_pairs)
        self.vocab = self.vocab_init(vocab_chrs, is_tgt)
        self.base_size = len(self.vocab)

    def vocab_init(self, vocab_chrs, is_tgt):
        vocab = {'<pad>': 0, '<unk>': 1, '<eos>': 2, '_': 3}
        if is_tgt:
            vocab['<bos>'] = 4
            
        uniq_toks = {tok for toks, _ in vocab_chrs for tok in toks}
        uniq_toks.remove('_')
        for i, tok in enumerate(uniq_toks, len(vocab)):
            vocab[tok] = i
        return vocab

    def pair_counts_init(self, vocab_pairs):
        pair_counts = defaultdict(int)
        for toks, freq in vocab_pairs:
            for pair in toks:
                pair_counts[pair] += freq
        return pair_counts

    def train_bpe(self):
        for i in tqdm(range(self.base_size, self.vocab_size)):
            pair = self.most_common_pair()
            self.vocab[pair] = i
            self.replace_pairs(pair)

    def replace_pairs(self, pair):
        pair_con = f"{pair[0]}{pair[1]}"
        p_1, p_2 = map(re.escape, pair)
        
        for i in range(len(self.vocab_chrs)):
            if pair in self.vocab_pairs[i][0]:
                new_chr = re.sub(rf"(?<=\s){p_1}\s{p_2}(?=\s)", pair_con, f" {' '.join(self.vocab_chrs[i][0])} ").split()
                self.vocab_chrs[i][0] = new_chr
                self.update_pair_count(list(zip(new_chr, new_chr[1:])), i)
        self.pair_counts.pop(pair)

    def update_pair_count(self, new_pairs, i):
        old_pairs, val = self.vocab_pairs[i]
        for tup in old_pairs:
            self.pair_counts[tup] -= val  
        for tup in new_pairs:
            self.pair_counts[tup] += val
        self.vocab_pairs[i][0] = new_pairs
      
    def most_common_pair(self):
        return max(self.pair_counts, key=self.pair_counts.get)

In [109]:
vocab_test5 = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
pair_vocab5 = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_test5]
tokenizer_test5 = BPETokenizer_5(vocab_test5, pair_vocab5, 8000, False)

t = time.time()
tokenizer_test5.train_bpe()
print(f"{time.time()-t:.2f} seconds")

  0%|          | 0/7945 [00:00<?, ?it/s]

169.12 seconds


#### 3.4 BPE Encoder propably should add to the BPE Class Later

In [145]:
tokenizer_test5.vocab_chrs[11500]

[['gi', 'll', 'ingham_'], 8]

In [122]:
dct_encoder = {"".join(k): v for k, v in tokenizer_test5.vocab.items()}
dct_encoder

{'<pad>': 0,
 '<unk>': 1,
 '<eos>': 2,
 '_': 3,
 'd': 4,
 'j': 5,
 '7': 6,
 't': 7,
 ':': 8,
 'f': 9,
 ']': 10,
 '-': 11,
 's': 12,
 '.': 13,
 '6': 14,
 '5': 15,
 'p': 16,
 'r': 17,
 '"': 18,
 '?': 19,
 'l': 20,
 'n': 21,
 ',': 22,
 ')': 23,
 "'": 24,
 '/': 25,
 '4': 26,
 'g': 27,
 'w': 28,
 '$': 29,
 'i': 30,
 'y': 31,
 '!': 32,
 'c': 33,
 'v': 34,
 '3': 35,
 '2': 36,
 'k': 37,
 'h': 38,
 'o': 39,
 'z': 40,
 'x': 41,
 '[': 42,
 'e': 43,
 'a': 44,
 'b': 45,
 'q': 46,
 '%': 47,
 '8': 48,
 'm': 49,
 '9': 50,
 'u': 51,
 '1': 52,
 '(': 53,
 '0': 54,
 'e_': 55,
 '._': 56,
 't_': 57,
 'th': 58,
 's_': 59,
 'ou': 60,
 'n_': 61,
 'd_': 62,
 ',_': 63,
 'er': 64,
 'y_': 65,
 'you': 66,
 'o_': 67,
 'in': 68,
 'you_': 69,
 'i_': 70,
 'an': 71,
 'l_': 72,
 'r_': 73,
 'the_': 74,
 'g_': 75,
 'at_': 76,
 '?_': 77,
 'a_': 78,
 'll_': 79,
 'to_': 80,
 'ing_': 81,
 'er_': 82,
 'ar': 83,
 'f_': 84,
 'it_': 85,
 'me_': 86,
 'on_': 87,
 'wh': 88,
 'm_': 89,
 'on': 90,
 'ow': 91,
 "'s_": 92,
 'is_': 93,
 'v

In [125]:
dct_encoder.get("running_", None)

1538

In [120]:
char_factor("running")

['r', 'u', 'n', 'n', 'i', 'n', 'g', '_']

In [162]:
def word_encoder(word, vocab_pairs, vocab_enc):
    word_proc = char_factor(word)
    word_enc = vocab_enc.get("".join(word_proc), None)
    if word_enc:
        return [word_enc]
    else:
        word_pairs = list(zip(word_proc, word_proc[1:]))
        for pair in list(vocab_pairs.keys())[55:]:
            if pair in word_pairs:
                p_1, p_2 = map(re.escape, pair)
                word_proc = re.sub(rf"(?<=\s){p_1}\s{p_2}(?=\s)", f"{pair[0]}{pair[1]}", f" {' '.join(word_proc)} ").split()
                word_pairs = list(zip(word_proc, word_proc[1:]))
        return [vocab_enc[x] for x in word_proc]

In [169]:
word_encoder("poland", tokenizer_test5.vocab, dct_encoder)

[227, 782]

In [171]:
tokenize_eng("I've been thinking about asking her out.")

['i', "'ve", 'been', 'thinking', 'about', 'asking', 'her', 'out', '.']

In [172]:
def sentence_encoder(snt, vocab_pairs, vocab_enc):
    snt_proc = tokenize_eng(snt)
    return [word_encoder(wrd, vocab_pairs, vocab_enc) for wrd in snt_proc]

In [173]:
sentence_encoder("I've been thinking about asking her out.", tokenizer_test5.vocab, dct_encoder)

[[70], [213], [301], [1071], [228], [1707], [248], [150], [56]]